In [ ]:
import mysql.connector

try:
    conexion = mysql.connector.connect(
        host     = "localhost",
        user     = "root",
        password = "tu_contraseña"
    )
    print("CONEXION EXITOSA!")
    print("Version MySQL:", conexion.get_server_info())
    conexion.close()

except mysql.connector.Error as e:
    print("ERROR al conectar:")
    print(e)

CONEXION EXITOSA!
Version MySQL: 8.0.45


C:\Users\firen\AppData\Local\Temp\ipykernel_5660\475697926.py:10: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  print("Version MySQL:", conexion.get_server_info())


In [ ]:
import pandas as pd
import mysql.connector

# ── Conectar a MySQL ──────────────────────────────
conexion = mysql.connector.connect(
    host     = "localhost",
    user     = "root",
    password = "tu_contraseña"
)
cursor = conexion.cursor()

# ── Crear la base de datos ────────────────────────
cursor.execute("CREATE DATABASE IF NOT EXISTS abc_corporation")
cursor.execute("USE abc_corporation")
print("Base de datos creada!")

# ── Tabla 1: empleados ────────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS empleados (
        EmployeeNumber    INT PRIMARY KEY,
        Age               INT,
        Gender            VARCHAR(10),
        MaritalStatus     VARCHAR(20),
        Education         INT,
        EducationField    VARCHAR(50),
        Attrition         INT,
        DistanceFromHome  INT
    )
""")
print("Tabla empleados creada!")

# ── Tabla 2: puestos ──────────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS puestos (
        EmployeeNumber          INT PRIMARY KEY,
        Department              VARCHAR(50),
        JobRole                 VARCHAR(50),
        JobLevel                INT,
        BusinessTravel          VARCHAR(30),
        OverTime                INT,
        YearsAtCompany          INT,
        YearsInCurrentRole      INT,
        YearsSinceLastPromotion INT,
        YearsWithCurrManager    INT,
        TotalWorkingYears       INT,
        NumCompaniesWorked      INT,
        TrainingTimesLastYear   INT,
        FOREIGN KEY (EmployeeNumber) REFERENCES empleados(EmployeeNumber)
    )
""")
print("Tabla puestos creada!")

# ── Tabla 3: satisfaccion ─────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS satisfaccion (
        EmployeeNumber           INT PRIMARY KEY,
        JobSatisfaction          INT,
        EnvironmentSatisfaction  INT,
        RelationshipSatisfaction INT,
        WorkLifeBalance          INT,
        JobInvolvement           INT,
        PerformanceRating        INT,
        FOREIGN KEY (EmployeeNumber) REFERENCES empleados(EmployeeNumber)
    )
""")
print("Tabla satisfaccion creada!")

# ── Tabla 4: compensacion ─────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS compensacion (
        EmployeeNumber     INT PRIMARY KEY,
        MonthlyIncome      INT,
        MonthlyRate        INT,
        DailyRate          INT,
        HourlyRate         INT,
        PercentSalaryHike  INT,
        StockOptionLevel   INT,
        FOREIGN KEY (EmployeeNumber) REFERENCES empleados(EmployeeNumber)
    )
""")
print("Tabla compensacion creada!")

print("\nTodo listo! BBDD y 4 tablas creadas en abc_corporation!")

Base de datos creada!
Tabla empleados creada!
Tabla puestos creada!
Tabla satisfaccion creada!
Tabla compensacion creada!

Todo listo! BBDD y 4 tablas creadas en abc_corporation!


In [4]:
import pandas as pd

# ── Cargar el CSV limpio ──────────────────────────
df = pd.read_csv("hr_clean.csv")
print(f"CSV cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

# ── Limpiar nulos antes de insertar ──────────────
# MySQL no acepta NaN — los convertimos a None
df = df.where(pd.notnull(df), None)

# ── Insertar en tabla: empleados ─────────────────
insertar_empleados = """
    INSERT IGNORE INTO empleados
        (EmployeeNumber, Age, Gender, MaritalStatus,
         Education, EducationField, Attrition, DistanceFromHome)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""
datos_empleados = df[[
    "EmployeeNumber", "Age", "Gender", "MaritalStatus",
    "Education", "EducationField", "Attrition", "DistanceFromHome"
]].values.tolist()

cursor.executemany(insertar_empleados, datos_empleados)
conexion.commit()
print(f"Empleados insertados: {cursor.rowcount} filas")

# ── Insertar en tabla: puestos ───────────────────
insertar_puestos = """
    INSERT IGNORE INTO puestos
        (EmployeeNumber, Department, JobRole, JobLevel,
         BusinessTravel, OverTime, YearsAtCompany,
         YearsInCurrentRole, YearsSinceLastPromotion,
         YearsWithCurrManager, TotalWorkingYears,
         NumCompaniesWorked, TrainingTimesLastYear)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
datos_puestos = df[[
    "EmployeeNumber", "Department", "JobRole", "JobLevel",
    "BusinessTravel", "OverTime", "YearsAtCompany",
    "YearsInCurrentRole", "YearsSinceLastPromotion",
    "YearsWithCurrManager", "TotalWorkingYears",
    "NumCompaniesWorked", "TrainingTimesLastYear"
]].values.tolist()

cursor.executemany(insertar_puestos, datos_puestos)
conexion.commit()
print(f"Puestos insertados: {cursor.rowcount} filas")

# ── Insertar en tabla: satisfaccion ──────────────
insertar_satisfaccion = """
    INSERT IGNORE INTO satisfaccion
        (EmployeeNumber, JobSatisfaction, EnvironmentSatisfaction,
         RelationshipSatisfaction, WorkLifeBalance,
         JobInvolvement, PerformanceRating)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""
datos_satisfaccion = df[[
    "EmployeeNumber", "JobSatisfaction", "EnvironmentSatisfaction",
    "RelationshipSatisfaction", "WorkLifeBalance",
    "JobInvolvement", "PerformanceRating"
]].values.tolist()

cursor.executemany(insertar_satisfaccion, datos_satisfaccion)
conexion.commit()
print(f"Satisfaccion insertada: {cursor.rowcount} filas")

# ── Insertar en tabla: compensacion ──────────────
insertar_compensacion = """
    INSERT IGNORE INTO compensacion
        (EmployeeNumber, MonthlyIncome, MonthlyRate,
         DailyRate, HourlyRate, PercentSalaryHike, StockOptionLevel)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""
datos_compensacion = df[[
    "EmployeeNumber", "MonthlyIncome", "MonthlyRate",
    "DailyRate", "HourlyRate", "PercentSalaryHike", "StockOptionLevel"
]].values.tolist()

cursor.executemany(insertar_compensacion, datos_compensacion)
conexion.commit()
print(f"Compensacion insertada: {cursor.rowcount} filas")

print("\nTodos los datos insertados correctamente!")

CSV cargado: 1470 filas x 32 columnas
Empleados insertados: 1470 filas
Puestos insertados: 1470 filas
Satisfaccion insertada: 1470 filas
Compensacion insertada: 1470 filas

Todos los datos insertados correctamente!


In [5]:
# ── Verificación final ────────────────────────────

# Contar filas en cada tabla
tablas = ["empleados", "puestos", "satisfaccion", "compensacion"]

print("VERIFICACION DE DATOS EN MYSQL:")
print("=" * 40)

for tabla in tablas:
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    total = cursor.fetchone()[0]
    print(f"  {tabla:20s} → {total} filas")

# Ver primeras 3 filas de empleados
print("\nPrimeras 3 filas de empleados:")
cursor.execute("SELECT * FROM empleados LIMIT 3")
for fila in cursor.fetchall():
    print(" ", fila)

print("\nBBDD lista para las consultas de la demo!")

VERIFICACION DE DATOS EN MYSQL:
  empleados            → 1470 filas
  puestos              → 1470 filas
  satisfaccion         → 1470 filas
  compensacion         → 1470 filas

Primeras 3 filas de empleados:
  (1, 41, 'Female', 'Single', 2, 'Life Sciences', 1, 1)
  (2, 49, 'Male', 'Married', 1, 'Life Sciences', 0, 8)
  (4, 37, 'Male', 'Single', 2, 'Other', 1, 2)

BBDD lista para las consultas de la demo!


In [6]:
# ── CONSULTAS PARA LA DEMO ────────────────────────

print("=" * 55)
print("  CONSULTAS ABC CORPORATION — FASE 4")
print("=" * 55)

# ── CONSULTA 1 — ¿Cuántos empleados se fueron? ───
print("\n1. DISTRIBUCION DE ATTRITION:")
cursor.execute("""
    SELECT 
        CASE WHEN Attrition = 1 THEN 'Se fue' ELSE 'Se quedo' END AS Estado,
        COUNT(*) AS Total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM empleados), 2) AS Porcentaje
    FROM empleados
    GROUP BY Attrition
""")
for fila in cursor.fetchall():
    print(f"  {fila[0]:12s} → {fila[1]} empleados ({fila[2]}%)")

# ── CONSULTA 2 — Tasa de abandono por departamento
print("\n2. TASA DE ABANDONO POR DEPARTAMENTO:")
cursor.execute("""
    SELECT 
        p.Department,
        COUNT(*) AS total,
        SUM(e.Attrition) AS se_fueron,
        ROUND(SUM(e.Attrition) * 100.0 / COUNT(*), 2) AS tasa_abandono
    FROM empleados e
    JOIN puestos p ON e.EmployeeNumber = p.EmployeeNumber
    GROUP BY p.Department
    ORDER BY tasa_abandono DESC
""")
for fila in cursor.fetchall():
    print(f"  {fila[0]:30s} → {fila[3]}% abandono")

# ── CONSULTA 3 — Rol con más abandono ────────────
print("\n3. TOP 5 ROLES CON MAYOR ABANDONO:")
cursor.execute("""
    SELECT 
        p.JobRole,
        COUNT(*) AS total,
        SUM(e.Attrition) AS se_fueron,
        ROUND(SUM(e.Attrition) * 100.0 / COUNT(*), 2) AS tasa_abandono
    FROM empleados e
    JOIN puestos p ON e.EmployeeNumber = p.EmployeeNumber
    GROUP BY p.JobRole
    ORDER BY tasa_abandono DESC
    LIMIT 5
""")
for fila in cursor.fetchall():
    print(f"  {fila[0]:30s} → {fila[3]}% abandono")

# ── CONSULTA 4 — Salario medio se fueron vs quedan
print("\n4. SALARIO MEDIO: SE FUERON VS SE QUEDARON:")
cursor.execute("""
    SELECT 
        CASE WHEN e.Attrition = 1 THEN 'Se fue' ELSE 'Se quedo' END AS Estado,
        ROUND(AVG(c.MonthlyIncome), 2) AS salario_medio,
        ROUND(MIN(c.MonthlyIncome), 2) AS salario_minimo,
        ROUND(MAX(c.MonthlyIncome), 2) AS salario_maximo
    FROM empleados e
    JOIN compensacion c ON e.EmployeeNumber = c.EmployeeNumber
    GROUP BY e.Attrition
""")
for fila in cursor.fetchall():
    print(f"  {fila[0]:12s} → Media: ${fila[1]:,.0f} | Min: ${fila[2]:,.0f} | Max: ${fila[3]:,.0f}")

# ── CONSULTA 5 — Impacto del overtime ────────────
print("\n5. IMPACTO DEL OVERTIME EN EL ABANDONO:")
cursor.execute("""
    SELECT 
        CASE WHEN p.OverTime = 1 THEN 'Con overtime' ELSE 'Sin overtime' END AS Overtime,
        COUNT(*) AS total,
        SUM(e.Attrition) AS se_fueron,
        ROUND(SUM(e.Attrition) * 100.0 / COUNT(*), 2) AS tasa_abandono
    FROM empleados e
    JOIN puestos p ON e.EmployeeNumber = p.EmployeeNumber
    GROUP BY p.OverTime
    ORDER BY tasa_abandono DESC
""")
for fila in cursor.fetchall():
    print(f"  {fila[0]:15s} → {fila[3]}% abandono")

# ── CONSULTA 6 — Coste estimado de la fuga ───────
print("\n6. COSTE MENSUAL ESTIMADO DE LA FUGA POR DEPARTAMENTO:")
cursor.execute("""
    SELECT 
        p.Department,
        SUM(e.Attrition) AS empleados_perdidos,
        ROUND(AVG(c.MonthlyIncome), 2) AS salario_medio,
        ROUND(SUM(e.Attrition) * AVG(c.MonthlyIncome), 2) AS coste_mensual
    FROM empleados e
    JOIN puestos p ON e.EmployeeNumber = p.EmployeeNumber
    JOIN compensacion c ON e.EmployeeNumber = c.EmployeeNumber
    GROUP BY p.Department
    ORDER BY coste_mensual DESC
""")
for fila in cursor.fetchall():
    print(f"  {fila[0]:30s} → {fila[1]} perdidos | Coste: ${fila[3]:,.0f}/mes")

# ── CONSULTA 7 — Perfil empleado en riesgo ───────
print("\n7. EMPLEADOS ACTUALES EN RIESGO DE FUGA:")
print("   (Junior + overtime + salario bajo + baja satisfaccion)")
cursor.execute("""
    SELECT 
        e.EmployeeNumber,
        p.Department,
        p.JobRole,
        c.MonthlyIncome,
        s.JobSatisfaction
    FROM empleados e
    JOIN puestos p ON e.EmployeeNumber = p.EmployeeNumber
    JOIN compensacion c ON e.EmployeeNumber = c.EmployeeNumber
    JOIN satisfaccion s ON e.EmployeeNumber = s.EmployeeNumber
    WHERE e.Attrition = 0
      AND p.JobLevel = 1
      AND p.OverTime = 1
      AND c.MonthlyIncome < 3000
      AND s.JobSatisfaction = 1
    ORDER BY c.MonthlyIncome ASC
    LIMIT 10
""")
filas = cursor.fetchall()
if filas:
    for fila in filas:
        print(f"  Emp#{fila[0]} | {fila[2]:30s} | ${fila[3]:,} | Satisf: {fila[4]}/4")
else:
    print("  No hay empleados en ese perfil de riesgo extremo")

print("\n" + "=" * 55)
print("  FIN DE LAS CONSULTAS")
print("=" * 55)

  CONSULTAS ABC CORPORATION — FASE 4

1. DISTRIBUCION DE ATTRITION:
  Se fue       → 237 empleados (16.12%)
  Se quedo     → 1233 empleados (83.88%)

2. TASA DE ABANDONO POR DEPARTAMENTO:
  Sales                          → 20.27% abandono
  Human Resources                → 19.05% abandono
  Research & Development         → 14.05% abandono

3. TOP 5 ROLES CON MAYOR ABANDONO:
  Sales Representative           → 39.76% abandono
  Laboratory Technician          → 23.94% abandono
  Human Resources                → 23.08% abandono
  Sales Executive                → 17.48% abandono
  Research Scientist             → 16.10% abandono

4. SALARIO MEDIO: SE FUERON VS SE QUEDARON:
  Se fue       → Media: $4,757 | Min: $1,009 | Max: $19,859
  Se quedo     → Media: $6,815 | Min: $1,051 | Max: $19,999

5. IMPACTO DEL OVERTIME EN EL ABANDONO:
  Con overtime    → 30.86% abandono
  Sin overtime    → 10.52% abandono

6. COSTE MENSUAL ESTIMADO DE LA FUGA POR DEPARTAMENTO:
  Research & Development         →

In [ ]:
import mysql.connector

# Cerramos cualquier conexión anterior
try:
    cursor.close()
    conexion.close()
    print("Conexion anterior cerrada!")
except:
    print("No habia conexion anterior")

# Abrimos conexion nueva
conexion = mysql.connector.connect(
    host     = "localhost",
    user     = "root",
    password = "tu_contraseña",
    database = "abc_corporation"
)
cursor = conexion.cursor()
print("Nueva conexion establecida!")

# Verificamos
tablas = ["empleados", "puestos", "satisfaccion", "compensacion"]
print("\nVERIFICACION:")
print("=" * 40)
for tabla in tablas:
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    total = cursor.fetchone()[0]
    print(f"  {tabla:20s} → {total} filas")

cursor.close()
conexion.close()
print("\nConexion cerrada correctamente!")

No habia conexion anterior
Nueva conexion establecida!

VERIFICACION:
  empleados            → 1470 filas
  puestos              → 1470 filas
  satisfaccion         → 1470 filas
  compensacion         → 1470 filas

Conexion cerrada correctamente!
